In [1]:
#Import + SageMaker session setup

In [1]:
import sagemaker
from sagemaker import Session
from sagemaker.inputs import TrainingInput

session = sagemaker.Session()
role = sagemaker.get_execution_role()


sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


In [2]:
# S3 paths to your already-prepared CSVs

In [3]:
bucket = "prognostica-cancer-project"
prefix = "features/"

train_path = f"s3://{bucket}/{prefix}lung_train.csv"
val_path   = f"s3://{bucket}/{prefix}lung_val.csv"
test_path  = f"s3://{bucket}/{prefix}lung_test.csv"


In [4]:
# Create TrainingInput objects

In [5]:
train_input = TrainingInput(train_path, content_type="text/csv")
val_input   = TrainingInput(val_path, content_type="text/csv")


In [6]:
# Create the XGBoost estimator (cheap CPU instance)

In [7]:
from sagemaker.amazon.amazon_estimator import get_image_uri
from sagemaker.estimator import Estimator

container = get_image_uri(
    session.boto_region_name,
    "xgboost",
    "1.5-1"
)
xgb = Estimator(
    image_uri=container,
    role=role,
    instance_count=1,
    instance_type="ml.m5.large",
    output_path=f"s3://{bucket}/{prefix}output"
)



The method get_image_uri has been renamed in sagemaker>=2.
See: https://sagemaker.readthedocs.io/en/stable/v2.html for details.


In [8]:
xgb.set_hyperparameters(
    objective="binary:logistic",
    num_round=200,
    max_depth=5,
    eta=0.2,
    subsample=0.8,
    eval_metric="auc"
)


In [9]:
#Train the model

In [10]:
xgb.fit({
    "train": train_input,
    "validation": val_input
})


INFO:sagemaker:Creating training-job with name: sagemaker-xgboost-2026-03-22-15-17-59-383


2026-03-22 15:18:00 Starting - Starting the training job...
2026-03-22 15:18:15 Starting - Preparing the instances for training...
2026-03-22 15:18:38 Downloading - Downloading input data...
2026-03-22 15:19:18 Downloading - Downloading the training image......
2026-03-22 15:20:30 Training - Training image download completed. Training in progress.
2026-03-22 15:20:30 Uploading - Uploading generated training model/miniconda3/lib/python3.8/site-packages/xgboost/compat.py:36: FutureWarning: pandas.Int64Index is deprecated and will be removed from pandas in a future version. Use pandas.Index with the appropriate dtype instead.
  from pandas import MultiIndex, Int64Index
[2026-03-22 15:20:23.416 ip-10-2-74-235.ec2.internal:7 INFO utils.py:28] RULE_JOB_STOP_SIGNAL_FILENAME: None
[2026-03-22 15:20:23.437 ip-10-2-74-235.ec2.internal:7 INFO profiler_config_parser.py:111] User has disabled profiler.
[2026-03-22:15:20:23:INFO] Imported framework sagemaker_xgboost_container.training
[2026-03-22:15

In [11]:
# The model artifact path

In [14]:
model_artifact = xgb.model_data
model_artifact


's3://prognostica-cancer-project/features/output/sagemaker-xgboost-2026-03-22-15-17-59-383/output/model.tar.gz'

In [15]:
# Download and extract the model

In [16]:
import tarfile
import boto3

s3 = boto3.client("s3")

# Parse bucket + key from the model artifact path
artifact_bucket = model_artifact.split("/")[2]
artifact_key = "/".join(model_artifact.split("/")[3:])

# Download locally
s3.download_file(artifact_bucket, artifact_key, "model.tar.gz")

# Extract
tar = tarfile.open("model.tar.gz")
tar.extractall()
tar.close()


/tmp/ipykernel_638/3517799135.py:15: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall()


In [17]:
# Load test dataset from S3

In [18]:
import pandas as pd

test_df = pd.read_csv(test_path)
test_df.head()


,gender,age,smoking,yellow_fingers,anxiety,peer_pressure,chronic_disease,fatigue,allergy,wheezing,alcohol_consuming,coughing,shortness_of_breath,swallowing_difficulty,chest_pain,lung_cancer,age_group_41-50,age_group_51-60,age_group_61-70,age_group_71+
0,M,59,1,1,2,1,1,2,2,1,2,1,2,2,2,YES,False,True,False,False
1,M,76,2,1,1,1,1,2,1,2,2,1,1,2,2,YES,False,False,False,True
2,M,43,1,2,1,2,2,2,2,2,1,2,1,1,2,YES,True,False,False,False
3,M,57,1,1,2,2,1,2,1,1,2,1,2,1,1,NO,False,True,False,False
4,M,71,2,2,2,2,2,2,2,2,1,1,1,1,2,YES,False,False,False,True


In [19]:
# Split features and the target

In [20]:
X_test = test_df.drop("lung_cancer", axis=1)
y_test = test_df["lung_cancer"].map({"YES": 1, "NO": 0})


In [21]:
# Load the trained XGBoost model locally

In [22]:
import xgboost as xgb

loaded_model = xgb.Booster()
loaded_model.load_model("xgboost-model")


In [23]:
# Convert all object columns to category and enable categorical mode

In [24]:
# Convert object columns to category
for col in X_test.columns:
    if X_test[col].dtype == "object":
        X_test[col] = X_test[col].astype("category")


In [25]:
# Create matrix

In [26]:
dtest = xgb.DMatrix(X_test, enable_categorical=True)

In [27]:
# Generate predictions

In [28]:
# 1. One-hot encode test set
X_test_encoded = pd.get_dummies(X_test)

# 2. Get the number of features the model expects
expected_num_features = loaded_model.num_features()

# 3. Trim test set to match expected number of features
X_test_trimmed = X_test_encoded.iloc[:, :expected_num_features]

# 4. Create DMatrix
dtest = xgb.DMatrix(X_test_trimmed)

# 5. Predict
y_pred_proba = loaded_model.predict(dtest)
y_pred = (y_pred_proba >= 0.5).astype(int)



In [29]:
# Evaluate the model

In [30]:
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix, classification_report

print("Accuracy:", accuracy_score(y_test, y_pred))
print("AUC:", roc_auc_score(y_test, y_pred_proba))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))


Accuracy: 0.1307901907356948
AUC: 0.5
Confusion Matrix:
 [[ 384    0]
 [2552    0]]

Classification Report:
               precision    recall  f1-score   support

           0       0.13      1.00      0.23       384
           1       0.00      0.00      0.00      2552

    accuracy                           0.13      2936
   macro avg       0.07      0.50      0.12      2936
weighted avg       0.02      0.13      0.03      2936



/opt/conda/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/conda/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/conda/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [31]:
# Rebalance thr training data
# load training CSV

In [32]:
# add to training notebook 
# train_df = pd.read_csv(train_path)


In [33]:
# split features and target
#X_train = train_df.drop("lung_cancer", axis=1)
#y_train = train_df["lung_cancer"].map({"YES": 1, "NO": 0})


In [34]:
# one hot encode before balncing
# X_train_encoded = pd.get_dummies(X_train)


In [35]:
# apply smote to balance the classes
!pip install imbalanced-learn


  Using cached imbalanced_learn-0.14.1-py3-none-any.whl.metadata (8.9 kB)
  Using cached sklearn_compat-0.1.5-py3-none-any.whl.metadata (20 kB)
Using cached imbalanced_learn-0.14.1-py3-none-any.whl (235 kB)
Using cached sklearn_compat-0.1.5-py3-none-any.whl (20 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [imbalanced-learn][imbalanced-learn]


In [36]:
import boto3

s3 = boto3.client("s3")
response = s3.list_objects_v2(Bucket=bucket, Prefix=prefix)

for obj in response.get("Contents", []):
    print(obj["Key"])


features/
features//lung_train_balanced.csv
features/lung_model_data.csv
features/lung_test.csv
features/lung_train.csv
features/lung_val.csv
features/output/sagemaker-xgboost-2026-03-21-19-41-18-281/debug-output/claim.smd
features/output/sagemaker-xgboost-2026-03-21-19-41-18-281/debug-output/collections/000000000/worker_0_collections.json
features/output/sagemaker-xgboost-2026-03-21-19-41-18-281/debug-output/events/000000000000/000000000000_worker_0.tfevents
features/output/sagemaker-xgboost-2026-03-21-19-41-18-281/debug-output/events/000000000010/000000000010_worker_0.tfevents
features/output/sagemaker-xgboost-2026-03-21-19-41-18-281/debug-output/events/000000000020/000000000020_worker_0.tfevents
features/output/sagemaker-xgboost-2026-03-21-19-41-18-281/debug-output/events/000000000030/000000000030_worker_0.tfevents
features/output/sagemaker-xgboost-2026-03-21-19-41-18-281/debug-output/events/000000000040/000000000040_worker_0.tfevents
features/output/sagemaker-xgboost-2026-03-21-19-

In [37]:
print(train_path)


s3://prognostica-cancer-project/features/lung_train.csv


In [38]:
# 1. Load training CSV from S3
train_df = pd.read_csv("s3://prognostica-cancer-project/features/lung_train_balanced.csv")


# 2. Split features and target
X_train = train_df.drop("lung_cancer", axis=1)
y_train = train_df["lung_cancer"].map({"YES": 1, "NO": 0})

# 3. One-hot encode before balancing
X_train_encoded = pd.get_dummies(X_train)

# 4. Apply SMOTE to balance the classes
from imblearn.over_sampling import SMOTE

sm = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = sm.fit_resample(X_train_encoded, y_train)

# 5. Save the balanced dataset locally
balanced_df = pd.concat([X_train_balanced, y_train_balanced], axis=1)
balanced_df.to_csv("lung_train_balanced.csv", index=False)

# 6. Upload to S3
session.upload_data(
    "lung_train_balanced.csv",
    bucket=bucket,
    key_prefix="features"
)

# 7. Update training path
train_path = f"s3://{bucket}/features/lung_train_balanced.csv"

# 8. Create S3 path in a  TrainingInput object
from sagemaker.inputs import TrainingInput

train_input = TrainingInput(
    train_path,
    content_type="text/csv"
)

# 9. Define XGBoost estimator
xgb = Estimator(
    image_uri=container,
    role=role,
    instance_count=1,
    instance_type="ml.m5.large",
    output_path=f"s3://{bucket}/{prefix}output"
)

# 10. Set hyperparameters
xgb.set_hyperparameters(
    objective="binary:logistic",
    num_round=500,
    max_depth=5,
    eta=0.2,
    subsample=0.8,
    eval_metric="auc",
    scale_pos_weight=1
)

# 11. Launch training job
xgb.fit({"train": train_input})


╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:2                                                                                    │
│                                                                                                  │
│    1 # 1. Load training CSV from S3                                                              │
│ ❱  2 train_df = pd.read_csv("s3://prognostica-cancer-project/features/lung_train_balanced.csv    │
│    3                                                                                             │
│    4                                                                                             │
│    5 # 2. Split features and target                                                              │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/pandas/io/parsers/readers.py:1026 in read_csv            │
│                                                                                                  │
│   1023 │   )                                                                                     │
│   1024 │   kwds.update(kwds_defaults)                                                            │
│   1025 │                                                                                         │
│ ❱ 1026 │   return _read(filepath_or_buffer, kwds)                                                │
│   1027                                                                                           │
│   1028                                                                                           │
│   1029 # iterator=True -> TextFileReader                                                         │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/pandas/io/parsers/readers.py:620 in _read                │
│                                                                                                  │
│    617 │   _validate_names(kwds.get("names", None))                                              │
│    618 │                                                                                         │
│    619 │   # Create the parser.                                                                  │
│ ❱  620 │   parser = TextFileReader(filepath_or_buffer, **kwds)                                   │
│    621 │                                                                                         │
│    622 │   if chunksize or iterator:                                                             │
│    623 │   │   return parser                                                                     │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/pandas/io/parsers/readers.py:1620 in __init__            │
│                                                                                                  │
│   1617 │   │   │   self.options["has_index_names"] = kwds["has_index_names"]                     │
│   1618 │   │                                                                                     │
│   1619 │   │   self.handles: IOHandles | None = None                                             │
│ ❱ 1620 │   │   self._engine = self._make_engine(f, self.engine)                                  │
│   1621 │                                                                                         │
│   1622 │   def close(self) -> None:                                                              │
│   1623 │   │   if self.handles is not None:                                                      │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/pandas/io/parsers/r

In [ ]:
response = s3.list_objects_v2(Bucket=bucket, Prefix="features")
for obj in response.get("Contents", []):
    print(obj["Key"])


In [ ]:
from imblearn.over_sampling import SMOTE

sm = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = sm.fit_resample(X_train_encoded, y_train)


In [ ]:
# save the balanced datset to s3

In [ ]:
balanced_df = pd.concat([X_train_balanced, y_train_balanced], axis=1)


In [ ]:
# save

In [ ]:
balanced_df.to_csv("lung_train_balanced.csv", index=False)


In [ ]:
#upload to a3
session.upload_data(
    "lung_train_balanced.csv",
    bucket=bucket,
    key_prefix=prefix
)


In [ ]:
#training path becomes...
train_path = f"s3://{bucket}/{prefix}lung_train_balanced.csv"


In [ ]:
# define estimator
xgb = Estimator(
    image_uri=container,
    role=role,
    instance_count=1,
    instance_type="ml.m5.large",
    output_path=f"s3://{bucket}/{prefix}output"
)


In [ ]:
# set hyperparameters
xgb.set_hyperparameters(
    objective="binary:logistic",
    num_round=500,
    max_depth=5,
    eta=0.2,
    subsample=0.8,
    eval_metric="auc",
    scale_pos_weight=1
)
